# IEEE-CIS Fraud Detection: Feature Engineering

**Goal:** Transform raw data into model-ready features based on EDA insights

**Key EDA Insights to Leverage:**
1. Email match is a strong fraud signal (9.6% vs 2.2%)
2. New cards (low D1) have higher fraud rates
3. Time-based patterns exist in hour/day
4. Mobile devices show higher fraud rates
5. V features cluster by missing patterns
6. Transaction amount patterns differ by fraud status

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

from utils import disp_columns

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)

## 1. Load Data

In [11]:
# Load training data
train_txn = pd.read_csv(DATA_DIR / 'train_transaction.csv')
train_id = pd.read_csv(DATA_DIR / 'train_identity.csv')

# Load test data
test_txn = pd.read_csv(DATA_DIR / 'test_transaction.csv')
test_id = pd.read_csv(DATA_DIR / 'test_identity.csv')

print(f"Train transactions: {train_txn.shape}")
print(f"Test transactions: {test_txn.shape}")

Train transactions: (590540, 394)
Test transactions: (506691, 393)


In [12]:
# Merge transaction and identity data
train = pd.merge(train_txn, train_id, on='TransactionID', how='left')
test = pd.merge(test_txn, test_id, on='TransactionID', how='left')

# Store target and IDs separately
y_train = train['isFraud'].copy()
train_ids = train['TransactionID'].copy()
test_ids = test['TransactionID'].copy()

print(f"Train shape after merge: {train.shape}")
print(f"Test shape after merge: {test.shape}")

Train shape after merge: (590540, 434)
Test shape after merge: (506691, 433)


In [13]:

print("Columns in train but not in test: ", len(set(train.columns) - set(test.columns)))
print("Columns in test but not in train: ", len(set(test.columns) - set(train.columns)))

disp_columns(train)
disp_columns(test)

Columns in train but not in test:  39
Columns in test but not in train:  38
┌────────────────────────────────────────────────┐
│ - TransactionID: int64                         │
│ - isFraud: int64                               │
│ - TransactionDT: int64                         │
│ - TransactionAmt: float64                      │
│ - ProductCD: object                            │
│ - card<N>                                      │
│   - N = 1: int64                               │
│   - N = 2-3, 5: float64                        │
│   - N = 4, 6: object                           │
│ - addr<1-2>: float64                           │
│ - dist<1-2>: float64                           │
│ - P_emaildomain: object                        │
│ - R_emaildomain: object                        │
│ - C<1-14>: float64                             │
│ - D<1-15>: float64                             │
│ - M<1-9>: object                               │
│ - V<1-339>: float64                            │
│ - id

## 2. Build Pipeline

### Column name normalization

The test data has columns `id-01`, `id-02`, etc. (with "-") whereas the train data has columns `id_01`, `id_02`, etc. (with "_"). We add an initial step (`ColumnNormalizer`) to the pipeline to normalize the column names.

### Encoding Strategy

Based on EDA analysis of feature distributions:

| Column(s) | Encoding | Reason |
|-----------|----------|--------|
| `card1`, `addr1` | Frequency | High cardinality, used to build composite customer UID |
| `id_01`, `id_03`–`id_06`, `id_09`–`id_11`, `id_13`, `id_14`, `id_17`–`id_22`, `id_24`–`id_26` | Frequency | Numeric dtype but categorical behavior (discrete peaks in distribution) |
| `id_32` | Label | Low cardinality (4 values: 0, 16, 24, 32 — likely screen color depth) |
| `id_02`, `id_07`, `id_08` | None (keep numeric) | True continuous distributions |
| `card4`, `card6`, `ProductCD`, `M1`–`M9`, `DeviceType`, etc. | Label | Already string dtype, picked up by `CategoricalEncoder` |

We will create two pipelines: one for lightGBM and tree-based methods, using label-encoding for categorical fields; and one using one-hot encoding for these fields which will be more suitable for regression techniques. 

### New features

#### Timestamp features

In order to utilize patterns in the `TransactionDT` timestamps we create new features with the `TimeFeatures` transformer to capture the hour-of-day (`hod_sin`, `hod_cos`) and day-of-week (`dow_sin`, `dow_cos`).

#### Email features

We also create new features based on the email addresses in `P_emaildomain` and `R_emaildomain`:
- `email_match`: whether the `P_emaildomain` and `R_emaildomain` are the same
- `P_email_is_free` and `R_email_is_free`: whether the domain is a free email domain
- `P_email_missing` and `R_email_missing`: whether the domain is missing

#### CardFeatures

- `is_new_card`: whether the card is new, `D1 <= 7`
- `has_identity`: whether the card is present in `identity`
- `is_mobile`: whether `DeviceType == 'mobile'`

#### Amount features

- `TransactionAmt_log`
- `TransactionAmt_decimal`
- `TransactionAmt_is_round`

#### Missing Indicators

- Boolean `D8_missing` etc. for the `D\d+` fields
- A count of how many `V\d+` fields are missing

In [14]:
from sklearn.pipeline import Pipeline

from transformers import (
    ColumnNormalizer,
    TimeFeatures,
    EmailFeatures,
    CardFeatures,
    AmountFeatures,
    AggregationFeatures,
    FrequencyEncoder,
    MissingIndicators,
    AsCategory,
    CategoricalEncoder,
    OneHotEncoder,
)

# Columns identified in EDA as needing special encoding:
# - High-cardinality: frequency encode (card1, addr1, most numeric id cols)
# - Low-cardinality categorical: convert to string for label encoding (id_32)
# - True numeric: leave as-is (id_02, id_07, id_08)

ID_FREQ_COLS = [
    'id_01', 'id_03', 'id_04', 'id_05', 'id_06', 'id_09', 'id_10', 'id_11',
    'id_13', 'id_14', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22',
    'id_24', 'id_25', 'id_26',
]

# Shared preprocessing steps (before categorical encoding)
SHARED_STEPS = [
    ('normalize_columns', ColumnNormalizer()),
    ('time', TimeFeatures()),
    ('email', EmailFeatures()),
    ('card', CardFeatures()),
    ('amount', AmountFeatures()),
    ('aggregation', AggregationFeatures(uid_cols=['card1', 'addr1'])),
    ('frequency', FrequencyEncoder(cols=['card1', 'addr1'] + ID_FREQ_COLS)),
    ('missing', MissingIndicators()),
    ('as_category', AsCategory(cols=['id_32'])),
]

# Pipeline for tree-based models (LightGBM, XGBoost)
# Uses label encoding - trees can handle arbitrary numeric splits
tree_pipeline = Pipeline(SHARED_STEPS + [
    ('categorical', CategoricalEncoder()),
])

# Pipeline for linear models (Logistic Regression, SVM)
# Uses one-hot encoding - linear models need proper categorical representation
linear_pipeline = Pipeline(SHARED_STEPS + [
    ('categorical', OneHotEncoder(max_categories=50)),
])

print("Tree pipeline (for LightGBM):")
for name, step in tree_pipeline.steps:
    print(f"  - {name}: {step.__class__.__name__}")

print("\nLinear pipeline (for Logistic Regression):")
for name, step in linear_pipeline.steps:
    print(f"  - {name}: {step.__class__.__name__}")

Tree pipeline (for LightGBM):
  - normalize_columns: ColumnNormalizer
  - time: TimeFeatures
  - email: EmailFeatures
  - card: CardFeatures
  - amount: AmountFeatures
  - aggregation: AggregationFeatures
  - frequency: FrequencyEncoder
  - missing: MissingIndicators
  - as_category: AsCategory
  - categorical: CategoricalEncoder

Linear pipeline (for Logistic Regression):
  - normalize_columns: ColumnNormalizer
  - time: TimeFeatures
  - email: EmailFeatures
  - card: CardFeatures
  - amount: AmountFeatures
  - aggregation: AggregationFeatures
  - frequency: FrequencyEncoder
  - missing: MissingIndicators
  - as_category: AsCategory
  - categorical: OneHotEncoder


## 3. Fit and Transform

In [15]:
# Fit and transform with TREE pipeline (for LightGBM)
train_tree = tree_pipeline.fit_transform(train)
test_tree = tree_pipeline.transform(test)

print(f"Tree pipeline - Train shape: {train_tree.shape}")
print(f"Tree pipeline - Test shape: {test_tree.shape}")

# Fit and transform with LINEAR pipeline (for Logistic Regression)
linear_pipeline_fitted = Pipeline(SHARED_STEPS + [
    ('categorical', OneHotEncoder(max_categories=50)),
])
train_linear = linear_pipeline_fitted.fit_transform(train)
test_linear = linear_pipeline_fitted.transform(test)

print(f"\nLinear pipeline - Train shape: {train_linear.shape}")
print(f"Linear pipeline - Test shape: {test_linear.shape}")
print(f"\nOne-hot encoding added {train_linear.shape[1] - train_tree.shape[1]} features")

Tree pipeline - Train shape: (590540, 457)
Tree pipeline - Test shape: (506691, 456)

Linear pipeline - Train shape: (590540, 522)
Linear pipeline - Test shape: (506691, 521)

One-hot encoding added 65 features


In [16]:
# Select features (drop ID, target, raw time)
drop_cols = ['TransactionID', 'isFraud', 'TransactionDT']

def select_features(train_df, test_df, drop_cols):
    """Select common features between train and test."""
    common_cols = set(train_df.columns) & set(test_df.columns)
    feature_cols = [c for c in train_df.columns if c in common_cols and c not in drop_cols]
    return train_df[feature_cols], test_df[feature_cols], feature_cols

# Tree pipeline features
X_train_tree, X_test_tree, tree_feature_cols = select_features(train_tree, test_tree, drop_cols)
print(f"Tree pipeline - Features: {len(tree_feature_cols)}")

# Linear pipeline features  
X_train_linear, X_test_linear, linear_feature_cols = select_features(train_linear, test_linear, drop_cols)
print(f"Linear pipeline - Features: {len(linear_feature_cols)}")

Tree pipeline - Features: 454
Linear pipeline - Features: 519


## 4. Save Processed Data and Pipeline

In [17]:
import joblib

# Save tree pipeline data (for LightGBM)
X_train_tree.to_parquet(PROCESSED_DIR / 'X_train_tree.parquet')
X_test_tree.to_parquet(PROCESSED_DIR / 'X_test_tree.parquet')

# Save linear pipeline data (for Logistic Regression)
X_train_linear.to_parquet(PROCESSED_DIR / 'X_train_linear.parquet')
X_test_linear.to_parquet(PROCESSED_DIR / 'X_test_linear.parquet')

# Save shared data
y_train.to_frame().to_parquet(PROCESSED_DIR / 'y_train.parquet')
test_ids.to_frame().to_parquet(PROCESSED_DIR / 'test_ids.parquet')

# Save pipelines for reuse
joblib.dump(tree_pipeline, PROCESSED_DIR / 'tree_pipeline.joblib')
joblib.dump(linear_pipeline_fitted, PROCESSED_DIR / 'linear_pipeline.joblib')

# Save feature lists
pd.Series(tree_feature_cols).to_csv(PROCESSED_DIR / 'tree_feature_cols.csv', index=False)
pd.Series(linear_feature_cols).to_csv(PROCESSED_DIR / 'linear_feature_cols.csv', index=False)

print(f"Saved processed data to {PROCESSED_DIR}:")
print(f"  - Tree pipeline: {X_train_tree.shape[1]} features")
print(f"  - Linear pipeline: {X_train_linear.shape[1]} features")

Saved processed data to ../data/processed:
  - Tree pipeline: 454 features
  - Linear pipeline: 519 features


## 5. Sanity Check

In [18]:
new_features = [
    # Time features
    'hod_sin', 'hod_cos', 'dow_sin', 'dow_cos',
    # Email/card features
    'email_match', 'P_email_is_free', 'is_new_card', 
    'has_identity', 'is_mobile', 'TransactionAmt_log',
    'v_missing_count',
    # UID-based aggregation features (card1 + addr1 composite)
    'card1_addr1_amt_mean', 'card1_addr1_amt_std', 
    'card1_addr1_txn_count', 'card1_addr1_amt_zscore',
    # Frequency encoding features
    'card1_freq', 'addr1_freq',
    'id_01_freq', 'id_05_freq', 'id_06_freq',
]

print("Shared feature statistics (tree pipeline):")
X_train_tree[new_features].describe().round(3)

Shared feature statistics (tree pipeline):


,hod_sin,hod_cos,dow_sin,dow_cos,email_match,P_email_is_free,is_new_card,has_identity,is_mobile,TransactionAmt_log,v_missing_count,card1_addr1_amt_mean,card1_addr1_amt_std,card1_addr1_txn_count,card1_addr1_amt_zscore,card1_freq,addr1_freq,id_01_freq,id_05_freq,id_06_freq
count,590540.000,590540.000,590540.000,590540.000,590540.000,590540.000,590540.000,590540.000,590540.000,590540.000,590540.000,590540.000,590540.000,590540.000,590540.000,590540.000,590540.000,590540.000,590540.000,590540.000
mean,-0.340,0.265,0.018,0.031,0.174,0.702,0.527,0.238,0.094,4.383,145.900,135.027,164.365,785.114,-0.000,0.004,0.034,0.021,0.025,0.024
std,0.621,0.654,0.723,0.690,0.379,0.457,0.499,0.426,0.292,0.937,40.191,108.951,147.895,1672.379,0.962,0.006,0.027,0.048,0.057,0.056
min,-1.000,-1.000,-1.000,-1.000,0.000,0.000,0.000,0.000,0.000,0.224,11.000,0.588,0.000,1.000,-4.126,0.000,0.000,0.000,0.000,0.000
25%,-0.891,-0.289,-0.775,-0.662,0.000,0.000,0.000,0.000,0.000,3.791,159.000,85.722,63.930,26.000,-0.491,0.000,0.010,0.000,0.000,0.000
50%,-0.545,0.450,-0.059,-0.021,0.000,1.000,1.000,0.000,0.000,4.245,159.000,114.595,140.096,125.000,-0.264,0.002,0.026,0.000,0.000,0.000
75%,0.113,0.873,0.706,0.686,0.000,1.000,1.000,0.000,0.000,4.836,170.000,150.691,219.667,698.000,0.144,0.005,0.068,0.000,0.000,0.000
max,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,10.372,274.000,4463.950,6547.819,9928.000,26.235,0.025,0.078,0.139,0.157,0.155
